# DATASET PREPARATION

**Note**: This section contains the code used to prepare the dataset for model training. 
All preprocessing steps in this section were executed on Google Colab using cloud-hosted datasets. 
To run this section locally, the dataset paths in the code must be modified to match your local directory structure, and the required libraries must be installed in your environment. 

In [ ]:
"""Kaggle datasets downloading and extracting"""

# Import the Kaggle API file 
from google.colab import files
files.upload()

# Modifying the file permission 
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# Downloading the datasets
!kaggle datasets download -d serenaraju/yawn-eye-dataset-new
!kaggle datasets download -d prasadvpatil/mrl-dataset
!kaggle datasets download -d ismailnasri20/driver-drowsiness-dataset-ddd

#Extracting the datasets
!unzip /content/yawn-eye-dataset-new.zip -d /content/yawn-eye-dataset-new
!unzip /content/mrl-dataset.zip -d /content/mrl-dataset
!unzip /content/driver-drowsiness-dataset-ddd.zip -d /content/driver-drowsiness-dataset-ddd

# Downloading and extracting the dlib face shape predictor landmark file
!wget http://dlib.net/files/shape_predictor_68_face_landmarks.dat.bz2
!bzip2 -dk shape_predictor_68_face_landmarks.dat.bz2

In [ ]:
# --- STAGE 1: DATA PREPARATION SCRIPT ---
# This script is configured for the four datasets you have selected.

# Importing the required libraries
import os
import cv2
import dlib
import shutil
import random

print("--- INITIAL SETUP ---")
print("Libraries for data preparation imported successfully.")

# --- Define Paths and Parameters ---
dataset_paths = {
    'ddd_open': '/content/datasets/train/Open_Eyes',
    'ddd_closed': '/content/datasets/train/Closed_Eyes',
    'nthu_alert': '/content/datasets/Driver Drowsiness Dataset (DDD)/Non Drowsy',
    'nthu_drowsy': '/content/datasets/Driver Drowsiness Dataset (DDD)/Drowsy',
    'yawn_yes': '/content/dataset_new/test/yawn',
    'yawn_no': '/content/dataset_new/test/no_yawn'
}

# Define output directories that will be created in /content/
BASE_OUTPUT_DIR = '/content/processed_data'
RAW_CLASS_DIR = os.path.join(BASE_OUTPUT_DIR, 'raw_classes')
FINAL_DATASET_DIR = os.path.join(BASE_OUTPUT_DIR, 'final_dataset_faces')
SPLIT_DATASET_DIR = os.path.join(BASE_OUTPUT_DIR, 'split_dataset')

CLASSES = ['Alert', 'Drowsy_Eyes_Closed', 'Drowsy_Yawning']
IMG_SIZE = 64
SPLIT_RATIO = (0.8, 0.1, 0.1) # 80% Train, 10% Validation, 10% Test

# --- Load dlib's pre-trained model FROM THE KAGGLE DATASET ---
print("\nLoading dlib shape predictor model from Kaggle dataset...")
predictor_path = "/content/shape_predictor_68_face_landmarks.dat"
detector = dlib.get_frontal_face_detector()
predictor = dlib.shape_predictor(predictor_path)
print("Dlib model loaded successfully.")

# --- STAGE 1.1: Sort raw images into master classes ---
print("\n--- STAGE 1.1: Sorting raw images into master classes ---")
if os.path.exists(RAW_CLASS_DIR): shutil.rmtree(RAW_CLASS_DIR)
for cls in CLASSES: os.makedirs(os.path.join(RAW_CLASS_DIR, cls), exist_ok=True)

def copy_images(src_dir, dest_dir):
    if not os.path.exists(src_dir):
        print(f"Warning: Source directory not found: {src_dir}")
        return
    for img_name in os.listdir(src_dir):
        # Ensure we are only copying image files
        if img_name.lower().endswith(('.png', '.jpg', '.jpeg')):
            shutil.copy(os.path.join(src_dir, img_name), dest_dir)

print("Populating 'Alert' class...")
copy_images(dataset_paths['ddd_open'], os.path.join(RAW_CLASS_DIR, 'Alert'))
copy_images(dataset_paths['nthu_alert'], os.path.join(RAW_CLASS_DIR, 'Alert'))

print("Populating 'Drowsy_Eyes_Closed' class...")
copy_images(dataset_paths['ddd_closed'], os.path.join(RAW_CLASS_DIR, 'Drowsy_Eyes_Closed'))
copy_images(dataset_paths['nthu_drowsy'], os.path.join(RAW_CLASS_DIR, 'Drowsy_Eyes_Closed'))

print("Populating 'Drowsy_Yawning' class...")
copy_images(dataset_paths['yawn_yes'], os.path.join(RAW_CLASS_DIR, 'Drowsy_Yawning'))
copy_images(dataset_paths['yawn_no'], os.path.join(RAW_CLASS_DIR, 'Drowsy_Yawning'))
print("Image sorting complete.")

In [ ]:
# --- STAGE 1.2: Pre-processing images (cropping, resizing, grayscaling) ---

print("\n--- STAGE 1.2: Pre-processing images (cropping, resizing, grayscaling) ---")
if os.path.exists(FINAL_DATASET_DIR): shutil.rmtree(FINAL_DATASET_DIR)
os.makedirs(FINAL_DATASET_DIR)

def process_and_save_images(source_class_dir, target_class_dir):
    os.makedirs(target_class_dir, exist_ok=True)
    images = [f for f in os.listdir(source_class_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
    total_images = len(images)
    print(f"Processing {total_images} images for class '{os.path.basename(source_class_dir)}'...")
    processed_count = 0
    for i, img_name in enumerate(images):
        img_path = os.path.join(source_class_dir, img_name)
        try:
            img = cv2.imread(img_path)
            if img is None: continue
            gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
            faces = detector(gray)
            if len(faces) > 0:
                face = faces[0]
                # Ensure coordinates are valid
                top, bottom, left, right = face.top(), face.bottom(), face.left(), face.right()
                if top < 0: top = 0
                if left < 0: left = 0
                if bottom > gray.shape[0]: bottom = gray.shape[0]
                if right > gray.shape[1]: right = gray.shape[1]

                cropped_face = gray[top:bottom, left:right]

                # Check if cropped face is not empty
                if cropped_face.size == 0:
                    continue

                resized_face = cv2.resize(cropped_face, (IMG_SIZE, IMG_SIZE))
                cv2.imwrite(os.path.join(target_class_dir, img_name), resized_face)
                processed_count += 1
        except Exception as e:
            print(f"Could not process image {img_name}: {e}")
        if (i + 1) % 500 == 0: print(f"  ...scanned {i+1}/{total_images} images.")
    print(f"  -> Successfully processed and saved {processed_count} face images.")


for cls in CLASSES:
    process_and_save_images(os.path.join(RAW_CLASS_DIR, cls), os.path.join(FINAL_DATASET_DIR, cls))
print("Image pre-processing complete.")

In [ ]:
# --- STAGE 1.3: Splitting dataset into train, validation, and test sets ---

print("\n--- STAGE 1.3: Splitting dataset into train, validation, and test sets ---")
if os.path.exists(SPLIT_DATASET_DIR): shutil.rmtree(SPLIT_DATASET_DIR)

for split in ['train', 'validation', 'test']:
    for cls in CLASSES:
        os.makedirs(os.path.join(SPLIT_DATASET_DIR, split, cls), exist_ok=True)

for cls in CLASSES:
    src_folder = os.path.join(FINAL_DATASET_DIR, cls)
    images = os.listdir(src_folder)
    random.shuffle(images)
    train_count = int(len(images) * SPLIT_RATIO[0])
    val_count = int(len(images) * SPLIT_RATIO[1])
    train_files, val_files, test_files = images[:train_count], images[train_count:train_count+val_count], images[train_count+val_count:]
    for f in train_files: shutil.copy(os.path.join(src_folder, f), os.path.join(SPLIT_DATASET_DIR, 'train', cls))
    for f in val_files: shutil.copy(os.path.join(src_folder, f), os.path.join(SPLIT_DATASET_DIR, 'validation', cls))
    for f in test_files: shutil.copy(os.path.join(src_folder, f), os.path.join(SPLIT_DATASET_DIR, 'test', cls))
    print(f"Class '{cls}': {len(train_files)} train, {len(val_files)} validation, {len(test_files)} test images.")

print("\n--- DATA PREPARATION COMPLETE! ---")
print(f"Your final, ready-to-use dataset is located in: {SPLIT_DATASET_DIR}")

In [ ]:
# --- STAGE 1.4: Counting the number of images under each training class folder ---

TRAIN_DIR = '/content/processed_data/split_dataset/train'

print("--- Checking the balance of our training data ---")

try:
    # Get the list of class folders
    class_folders = os.listdir(TRAIN_DIR)

    for cls in class_folders:
        class_path = os.path.join(TRAIN_DIR, cls)
        num_images = len(os.listdir(class_path))
        print(f"Class '{cls}': {num_images} training images.")

except FileNotFoundError:
    print("Error: The training directory was not found.")
    print("Please make sure you have successfully run the data preparation script first.")

# MODEL TRAINING

**Note**: This section contains the CNN model definition, training loop, and evaluation steps including the Classification Report and Confusion Matrix. The model was trained on Google Colab using the preprocessed dataset generated in the previous section. To run this section locally, ensure the 
dataset path points to your local preprocessed image folders. The model is configured to run on CPU to ensure compatibility across environments without requiring a GPU.

In [ ]:
""" --- STAGE 2: CNN MODEL TRAINING --- """

# Importing the required models
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, Input
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
import os

print("--- INITIAL SETUP ---")
print("Libraries for training and evaluation imported successfully.")

# --- Define Paths and Parameters ---
DATASET_DIR = '/content/processed_data/split_dataset'
TRAIN_DIR = os.path.join(DATASET_DIR, 'train')
VALIDATION_DIR = os.path.join(DATASET_DIR, 'validation')
TEST_DIR = os.path.join(DATASET_DIR, 'test')

IMG_SIZE = 64
BATCH_SIZE = 32
EPOCHS = 15

# --- Load and Prepare Data Generators ---
print("\n--- Preparing Data Generators ---")
train_datagen = ImageDataGenerator(rescale=1./255)
validation_datagen = ImageDataGenerator(rescale=1./255)
test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    color_mode='grayscale'
)

validation_generator = validation_datagen.flow_from_directory(
    VALIDATION_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    color_mode='grayscale'
)

test_generator = test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    color_mode='grayscale',
    shuffle=False
)

# --- Define and Train the Model on CPU ---
# Used 'with tf.device('/CPU:0')' to ensure all operations happen on the CPU
with tf.device('/CPU:0'):
    print("\n--- Building the CNN Model (on CPU) ---")
    model = Sequential([
        Input(shape=(IMG_SIZE, IMG_SIZE, 1)),
        Conv2D(16, (3, 3), activation='relu'), MaxPooling2D(2, 2),
        Conv2D(32, (3, 3), activation='relu'), MaxPooling2D(2, 2),
        Conv2D(64, (3, 3), activation='relu'), MaxPooling2D(2, 2),
        Flatten(),
        Dense(64, activation='relu'),
        Dropout(0.5),
        Dense(3, activation='softmax')
    ])

    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    model.summary()

    # --- Train the Model ---
    print("\n--- Starting Model Training (on CPU) ---")
    history = model.fit(
        train_generator,
        epochs=EPOCHS,
        validation_data=validation_generator
    )
    print("\n--- Model Training Complete ---")

# --- Save the Final Trained Model ---
print("\n--- Saving the Trained Model ---")
model.save('drowsiness_detection_model_v1.keras')
print("Model saved successfully as 'drowsiness_detection_model_v1.keras'")


# --- DETAILED EVALUATION ON THE TEST SET (on CPU) ---
print("\n--- Generating Detailed Performance Evaluation (on CPU) ---")
with tf.device('/CPU:0'):
    # 1. Get Predictions
    Y_pred_probs = model.predict(test_generator)

# Continue with the rest of the evaluation
y_pred = np.argmax(Y_pred_probs, axis=1)
y_true = test_generator.classes
class_labels = list(test_generator.class_indices.keys())

# 2. Generate and Print Classification Report
print("\n--- Classification Report ---")
print(classification_report(y_true, y_pred, target_names=class_labels, labels=np.arange(len(class_labels))))

# 3. Generate and Plot Confusion Matrix
print("\n--- Confusion Matrix ---")
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_labels, yticklabels=class_labels)
plt.title('Confusion Matrix')
plt.ylabel('Actual Class')
plt.xlabel('Predicted Class')
plt.show()


# --- Plot Training History ---
print("\n--- Generating Training History Plots ---")
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Model Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Model Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.show()

In [ ]:
""" --- STAGE 2.2: RE-TRAIN CNN WITH CLASS WEIGHTS FOR BALANCE ---
This script calculates and applies class weights to fix the imbalance problem 
and train a model that can recognize all three classes."""

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, Input
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight
import os

print("--- INITIAL SETUP ---")
print("Libraries for training and evaluation imported successfully.")

# --- Define Paths and Parameters ---
DATASET_DIR = '/content/processed_data/split_dataset'
TRAIN_DIR = os.path.join(DATASET_DIR, 'train')
VALIDATION_DIR = os.path.join(DATASET_DIR, 'validation')
TEST_DIR = os.path.join(DATASET_DIR, 'test')

IMG_SIZE = 64
BATCH_SIZE = 32
EPOCHS = 15

# --- Load and Prepare Data Generators ---
print("\n--- Preparing Data Generators ---")
train_datagen = ImageDataGenerator(rescale=1./255)
validation_datagen = ImageDataGenerator(rescale=1./255)
test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    color_mode='grayscale'
)

validation_generator = validation_datagen.flow_from_directory(
    VALIDATION_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    color_mode='grayscale'
)

test_generator = test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    color_mode='grayscale',
    shuffle=False
)

# --- NEW SECTION: CALCULATE CLASS WEIGHTS TO HANDLE IMBALANCE ---
print("\n--- Calculating Class Weights for a Balanced Model ---")
# Get the class labels from the generator
class_labels = train_generator.classes
# Use scikit-learn to calculate weights that will balance the dataset
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(class_labels),
    y=class_labels
)
# Converting the weights into a dictionary format that TensorFlow understands
class_weights_dict = {i: weight for i, weight in enumerate(class_weights)}
print("Class weights calculated:")
print(train_generator.class_indices)
print(class_weights_dict)

# --- Defining the Custom Lightweight CNN Model ---
print("\n--- Building the CNN Model ---")
# Using a CPU-only block to ensure the script is robust
with tf.device('/CPU:0'):
    model = Sequential([
        Input(shape=(IMG_SIZE, IMG_SIZE, 1)),
        Conv2D(16, (3, 3), activation='relu'), MaxPooling2D(2, 2),
        Conv2D(32, (3, 3), activation='relu'), MaxPooling2D(2, 2),
        Conv2D(64, (3, 3), activation='relu'), MaxPooling2D(2, 2),
        Flatten(),
        Dense(64, activation='relu'),
        Dropout(0.5),
        Dense(3, activation='softmax')
    ])

    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    model.summary()

    # --- Training the Model (NOW WITH CLASS WEIGHTS) ---
    print("\n--- Starting Model Training (with Class Weights) ---")
    history = model.fit(
        train_generator,
        epochs=EPOCHS,
        validation_data=validation_generator,
        class_weight=class_weights_dict  # ** THIS IS THE IMPORTANT CHANGE **
    )
    print("\n--- Model Training Complete ---")

# --- Save the Final Trained Model ---
print("\n--- Saving the Trained Model (V2) ---")
model.save('drowsiness_detection_model_v2.keras')
print("Model saved successfully as 'drowsiness_detection_model_v2.keras'")


# --- Detailed Evaluation ---
print("\n--- Generating Detailed Performance Evaluation ---")
with tf.device('/CPU:0'):
    Y_pred_probs = model.predict(test_generator)

y_pred = np.argmax(Y_pred_probs, axis=1)
y_true = test_generator.classes
class_labels_map = list(test_generator.class_indices.keys())

print("\n--- Classification Report ---")
print(classification_report(y_true, y_pred, target_names=class_labels_map, labels=np.arange(len(class_labels_map))))

print("\n--- Confusion Matrix ---")
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_labels_map, yticklabels=class_labels_map)
plt.title('Confusion Matrix')
plt.ylabel('Actual Class')
plt.xlabel('Predicted Class')
plt.show()

In [ ]:
# Saving the model as TF SavedModel format for compatibility across different versions.
# Also saving the weights seperately as a backup

model = tf.keras.models.load_model('/content/drowsiness_detection_model_v2.keras')
model.export('drowsiness_model_saved')

model.save_weights('drowsiness_model_weights.weights.h5')
print("Weights saved to 'drowsiness_model_weights.weights.h5'")

# Print model architecture so it can be rebuild locally if needed
model.summary()

# IMPLEMENTATION ACROSS DIFFERENT IMAGES

This section shows the code which was used to test the model across few images before utilising webcam.

In [ ]:
# Importing libraries
import cv2
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import dlib

model = tf.keras.models.load_model(
    "/content/drowsiness_detection_model_v2.keras",
    compile=False
)

classes = [
    "Alert",
    "Drowsy_Eyes_Closed",
    "Drowsy_Yawning"
]

# DLIB FACE DETECTOR
detector = dlib.get_frontal_face_detector()

image_path = "/content/test1.jpeg"
frame = cv2.imread(image_path)
gray = cv2.cvtColor(
    frame,
    cv2.COLOR_BGR2GRAY
)

faces = detector(gray)

for face in faces:

    x1 = face.left()
    y1 = face.top()

    x2 = face.right()
    y2 = face.bottom()

    face_crop = gray[y1:y2, x1:x2]
    resized = cv2.resize(
        face_crop,
        (64,64)
    )

    normalized = resized / 255.0
    reshaped = normalized.reshape(
        1,64,64,1
    )

    prediction = model.predict(
        reshaped,
        verbose=0
    )[0]

    class_index = np.argmax(prediction)
    label = classes[class_index]
    confidence = prediction[class_index]

    # Draw
    cv2.rectangle(
        frame,
        (x1,y1),
        (x2,y2),
        (0,255,0),
        2
    )
    cv2.putText(
        frame,
        f"{label}: {confidence:.2f}",
        (x1,y1-10),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.8,
        (0,255,0),
        2
    )

# Display result
plt.figure(figsize=(8,8))
plt.imshow(
    cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
)

plt.axis("off")
plt.show()

# WEBCAM IMPLEMENTATION

This section provides the script used to test the model using live webcam video.

In [ ]:
# IMPORTING THE REQUIRED LIBRARIES
import cv2
import time
import numpy as np
import mediapipe as mp
import tensorflow as tf
import os
import urllib.request

# CONFIGURATION
IMG_SIZE = 64
CLASSES = ["Alert", "Drowsy_Eyes_Closed", "Drowsy_Yawning"]
MODEL_PATH = "Drowsiness_model"          
WEIGHTS_PATH = "drowsiness_model_weights.weights.h5"  
LANDMARKER_PATH = "face_landmarker.task"

# AUTO-DOWNLOAD face_landmarker.task IF MISSING
if not os.path.exists(LANDMARKER_PATH):
    print("Downloading face_landmarker.task...")
    url = (
        "https://storage.googleapis.com/mediapipe-models/"
        "face_landmarker/face_landmarker/float16/1/face_landmarker.task"
    )
    urllib.request.urlretrieve(url, LANDMARKER_PATH)
    print("Downloaded face_landmarker.task")

# LOAD MODEL
print("Loading model...")

def build_model():
    """Rebuild the exact same architecture used in training."""
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, Input
    model = Sequential([
        Input(shape=(IMG_SIZE, IMG_SIZE, 1)),
        Conv2D(16, (3, 3), activation='relu'), MaxPooling2D(2, 2),
        Conv2D(32, (3, 3), activation='relu'), MaxPooling2D(2, 2),
        Conv2D(64, (3, 3), activation='relu'), MaxPooling2D(2, 2),
        Flatten(),
        Dense(64, activation='relu'),
        Dropout(0.5),
        Dense(3, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model

if os.path.exists(MODEL_PATH):
    try:
        model = tf.saved_model.load(MODEL_PATH)
        # Wrap for easy prediction
        infer = model.signatures["serving_default"]
        USE_SAVED_MODEL = True
        print(f"Loaded SavedModel from '{MODEL_PATH}'")
    except Exception as e:
        print(f"SavedModel load failed: {e}")
        USE_SAVED_MODEL = False
elif os.path.exists(WEIGHTS_PATH):
    # Fallback: rebuild architecture and load weights
    model = build_model()
    model.load_weights(WEIGHTS_PATH)
    USE_SAVED_MODEL = False
    print(f"Loaded weights from '{WEIGHTS_PATH}'")
else:
    #Try loading keras directly
    try:
        model = tf.keras.models.load_model(
            'drowsiness_detection_model_v2.keras',
            compile=False  # skip compile to avoid config issues
        )
        USE_SAVED_MODEL = False
        print("Loaded .keras model with compile=False")
    except Exception as e:
        print(f"ERROR: Could not load model. {e}")
        print("Please run colab_export_model.py on Colab first.")
        exit(1)

def predict(frame_input):
    """Run inference regardless of model type."""
    if USE_SAVED_MODEL:
        tensor = tf.constant(frame_input, dtype=tf.float32)
        # Auto-detect the input key name
        input_key = list(infer.structured_input_signature[1].keys())[0]
        result = infer(**{input_key: tensor})
        # Auto-detect the output key name 
        output_key = list(result.keys())[0]
        return result[output_key].numpy()[0]
    else:
        return model.predict(frame_input, verbose=0)[0]

print("Model ready.")

# MEDIAPIPE FACE LANDMARKER SETUP 
print("Initialising MediaPipe Face Landmarker...")

BaseOptions = mp.tasks.BaseOptions
FaceLandmarker = mp.tasks.vision.FaceLandmarker
FaceLandmarkerOptions = mp.tasks.vision.FaceLandmarkerOptions
RunningMode = mp.tasks.vision.RunningMode

options = FaceLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=LANDMARKER_PATH),
    running_mode=RunningMode.VIDEO,
    num_faces=1,
    output_face_blendshapes=False,
    output_facial_transformation_matrixes=False,
    min_face_detection_confidence=0.5,
    min_face_presence_confidence=0.5,
    min_tracking_confidence=0.5
)

detector = FaceLandmarker.create_from_options(options)
print("MediaPipe ready.")

# COLOUR MAP FOR LABELS
LABEL_COLORS = {
    "Alert":              (0, 220, 100),   # green
    "Drowsy_Eyes_Closed": (0, 100, 255),   # orange-red
    "Drowsy_Yawning":     (0, 200, 255),   # yellow
}

print("Opening webcam... Press Q to quit.")
cap = cv2.VideoCapture(0)

if not cap.isOpened():
    print("ERROR: Could not open webcam.")
    exit(1)

# Smoothing
from collections import deque, Counter
pred_history = deque(maxlen=15) 

LABEL_LOCK_SECONDS = 1.0
locked_label    = None
locked_color    = (180, 180, 180)
lock_until_time = 0.0

# Confidence threshold 
CONFIDENCE_THRESHOLD = 0.75
frame_ts = 0  

while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame = cv2.flip(frame, 1)
    h, w = frame.shape[:2]
    frame_ts += 33  

    # MediaPipe detection
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
    result = detector.detect_for_video(mp_image, frame_ts)

    label = "No Face"
    color = (180, 180, 180)

    if result.face_landmarks:
        landmarks = result.face_landmarks[0]

        xs = [int(lm.x * w) for lm in landmarks]
        ys = [int(lm.y * h) for lm in landmarks]

        for x, y in zip(xs, ys):
            cv2.circle(frame, (x, y), 1, (0, 255, 100), -1)

        x_min = max(min(xs) - 20, 0)
        y_min = max(min(ys) - 30, 0)
        x_max = min(max(xs) + 20, w)
        y_max = min(max(ys) + 20, h)

        face_crop = frame[y_min:y_max, x_min:x_max]

        if face_crop.size != 0:
            gray      = cv2.cvtColor(face_crop, cv2.COLOR_BGR2GRAY)
            resized   = cv2.resize(gray, (IMG_SIZE, IMG_SIZE))
            normalized = resized / 255.0
            reshaped  = normalized.reshape(1, IMG_SIZE, IMG_SIZE, 1).astype(np.float32)

            # Predict
            probs = predict(reshaped)
            class_idx = int(np.argmax(probs))
            label = CLASSES[class_idx]
            confidence = probs[class_idx]

            pred_history.append(class_idx)
            smooth_idx   = Counter(pred_history).most_common(1)[0][0]
            smooth_label = CLASSES[smooth_idx]

            now = time.time()
            if confidence < CONFIDENCE_THRESHOLD:
                smooth_label = "Uncertain"
                color        = (180, 180, 180)
            else:
                color = LABEL_COLORS[smooth_label]

            if now >= lock_until_time:
                locked_label    = smooth_label
                locked_color    = color
                lock_until_time = now + LABEL_LOCK_SECONDS

            display_label = locked_label if locked_label else smooth_label
            display_color = locked_color

            cv2.rectangle(frame, (x_min, y_min), (x_max, y_max), display_color, 2)

            main_text = f"{display_label}  {confidence:.0%}"
            cv2.putText(frame, main_text, (x_min, y_min - 12),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.75, display_color, 2)

            for i, cls in enumerate(CLASSES):
                bar_label = f"{cls}: {probs[i]:.0%}"
                bar_color = LABEL_COLORS[cls]
                cv2.putText(frame, bar_label, (10, 30 + i * 28),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, bar_color, 2)

    else:
        cv2.putText(frame, "No face detected", (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (180, 180, 180), 2)

    # FPS counter
    cv2.putText(frame, "Press Q to quit", (w - 160, h - 10),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (200, 200, 200), 1)

    cv2.imshow("Drowsiness Detection", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
detector.close()
cv2.destroyAllWindows()
print("Done.")